# Phase 31 — BM25 + SVM

BM25 (Best Match 25) είναι ένα ranking function που χρησιμοποιείται ευρέως
στο Information Retrieval (π.χ. Elasticsearch, Lucene).

**Διαφορά BM25 vs TF-IDF:**
```
TF-IDF: score = tf * idf
BM25:   score = idf * (tf * (k1+1)) / (tf + k1 * (1 - b + b * doc_len/avg_len))
```

Το BM25 διαφέρει σε δύο σημεία:
1. **Saturation:** Υψηλή συχνότητα λέξης δεν αυξάνει το score γραμμικά (k1 parameter)
2. **Length normalization:** Κανονικοποίηση βάσει μήκους εγγράφου (b parameter)

**Τυπικές τιμές:** k1=1.5, b=0.75

In [1]:
#!pip install rank_bm25 -q

In [10]:
import numpy as np
import pandas as pd
import re
import scipy.sparse as sp
from collections import Counter
from rank_bm25 import BM25Okapi
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

In [11]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

def preprocess(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def make_text(df):
    return (df['title'].apply(preprocess) + ' ' +
            df['text'].str[:550].apply(preprocess)).tolist()

texts_train = make_text(train)
texts_valid = make_text(valid)
texts_test  = make_text(test)

tokens_train = [t.split() for t in texts_train]
tokens_valid = [t.split() for t in texts_valid]
tokens_test  = [t.split() for t in texts_test]

print(f'Train: {len(train)} | Valid: {len(valid)} | Test: {len(test)}')

Train: 5082 | Valid: 565 | Test: 997


In [13]:
def official_st1_score(y_true_hazard, y_pred_hazard,
                       y_true_product, y_pred_product, verbose=True):
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard, average='macro', zero_division=0)
    mask = (np.array(y_true_hazard) == np.array(y_pred_hazard))
    f1_product = f1_score(
        np.array(y_true_product)[mask],
        np.array(y_pred_product)[mask],
        average='macro', zero_division=0
    ) if mask.sum() > 0 else 0.0
    score = (f1_hazard + f1_product) / 2
    if verbose:
        print(f'  macro-F1 Hazard:                    {f1_hazard:.4f}')
        print(f'  Σωστά hazard:                       {mask.sum()}/{len(mask)} ({100*mask.mean():.1f}%)')
        print(f'  macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'  OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score

In [14]:
# υλοποίηση BM25 ως TF-IDF variant
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# BM25 = TF-IDF με διαφορετικό TF weighting
# Χρησιμοποιούμε sublinear_tf=False και norm=None
# και μετά εφαρμόζουμε BM25 normalization χειροκίνητα

k1 = 1.5
b  = 0.75

# Raw TF (χωρίς normalization)
tf_vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 1),
    use_idf=True,
    sublinear_tf=False,
    norm=None,
    stop_words='english'
)

X_train_tf = tf_vectorizer.fit_transform(texts_train)
X_valid_tf = tf_vectorizer.transform(texts_valid)
X_test_tf  = tf_vectorizer.transform(texts_test)

# Doc lengths
doc_len_train = np.array(X_train_tf.sum(axis=1)).flatten()
doc_len_valid = np.array(X_valid_tf.sum(axis=1)).flatten()
doc_len_test  = np.array(X_test_tf.sum(axis=1)).flatten()
avg_len = doc_len_train.mean()
print(f'Average doc length: {avg_len:.1f}')

# BM25 TF normalization: tf_bm25 = tf * (k1+1) / (tf + k1*(1-b+b*dl/avgdl))
def bm25_transform(X, doc_lengths, avg_len, k1, b):
    X = X.astype(float)
    # Για κάθε non-zero element
    cx = X.tocsr()
    for i in range(cx.shape[0]):
        start = cx.indptr[i]
        end   = cx.indptr[i+1]
        tf    = cx.data[start:end]
        dl    = doc_lengths[i]
        cx.data[start:end] = tf * (k1+1) / (tf + k1*(1 - b + b*dl/avg_len))
    return cx

print('Εφαρμογή BM25 weighting...')
X_train_bm25 = bm25_transform(X_train_tf, doc_len_train, avg_len, k1, b)
X_valid_bm25 = bm25_transform(X_valid_tf, doc_len_valid, avg_len, k1, b)
X_test_bm25  = bm25_transform(X_test_tf,  doc_len_test,  avg_len, k1, b)
print(f'BM25 matrix shape: {X_train_bm25.shape}')

Average doc length: 267.1
Εφαρμογή BM25 weighting...
BM25 matrix shape: (5082, 12695)


In [15]:
print('=== SVM με BM25 Features ===')

clf_h = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_h.fit(X_train_bm25, train['hazard-category'])

clf_p = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_p.fit(X_train_bm25, train['product-category'])

score_bm25 = official_st1_score(
    valid['hazard-category'], clf_h.predict(X_valid_bm25),
    valid['product-category'], clf_p.predict(X_valid_bm25)
)

pd.DataFrame({
    'id': test['id'],
    'hazard-category':  clf_h.predict(X_test_bm25),
    'product-category': clf_p.predict(X_test_bm25)
}).to_csv('submission_bm25.csv', index=False)
print('\nΑποθηκεύτηκε: submission_bm25.csv')

=== SVM με BM25 Features ===
  macro-F1 Hazard:                    0.7595
  Σωστά hazard:                       518/565 (91.7%)
  macro-F1 Product (given correct h): 0.6793
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  OFFICIAL ST1 SCORE:                 0.7194

Αποθηκεύτηκε: submission_bm25.csv


In [16]:
# TF-IDF για σύγκριση
tfidf = TfidfVectorizer(
    max_features=50000, ngram_range=(1,2),
    sublinear_tf=True, stop_words='english'
)
X_train_tfidf = tfidf.fit_transform(texts_train)
X_valid_tfidf = tfidf.transform(texts_valid)

clf_h_t = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_h_t.fit(X_train_tfidf, train['hazard-category'])
clf_p_t = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_p_t.fit(X_train_tfidf, train['product-category'])

score_tfidf = official_st1_score(
    valid['hazard-category'], clf_h_t.predict(X_valid_tfidf),
    valid['product-category'], clf_p_t.predict(X_valid_tfidf),
    verbose=False
)

print('\n' + '='*50)
print('ΣΥΓΚΡΙΣΗ BM25 vs TF-IDF (validation)')
print('='*50)
print(f'TF-IDF + SVM:  {score_tfidf:.4f}')
print(f'BM25 + SVM:    {score_bm25:.4f}')
diff = score_bm25 - score_tfidf
print(f'Διαφορά:       {diff:+.4f}  ({"BM25 καλύτερο " if diff > 0 else "TF-IDF καλύτερο"})')

print('\n=== ΣΥΝΟΨΗ CLASSICAL ΜΕΘΟΔΩΝ ===')
results = [
    ('Naive Bayes',          0.3122),
    ('Random Forest',        0.4775),
    ('Word2Vec + SVM',       0.4629),
    ('GloVe + SVM',          0.4618),
    ('FastText + SVM',       0.4716),
    ('Logistic Regression',  0.6523),
    ('TF-IDF + SVM',         score_tfidf),
    ('BM25 + SVM',           score_bm25),
]
for name, score in sorted(results, key=lambda x: x[1]):
    bar = '█' * int(score * 35)
    print(f'  {name:22s} {score:.4f}  {bar}')
print(f'\n  {"--- Transformers ---"}')
print(f'  {"DistilBERT":22s} 0.7606')
print(f'  {"BERT-base Focal":22s} 0.8040')
print(f'  {"Best Ensemble":22s} 0.8188')


ΣΥΓΚΡΙΣΗ BM25 vs TF-IDF (validation)
TF-IDF + SVM:  0.7436
BM25 + SVM:    0.7194
Διαφορά:       -0.0243  (TF-IDF καλύτερο)

=== ΣΥΝΟΨΗ CLASSICAL ΜΕΘΟΔΩΝ ===
  Naive Bayes            0.3122  ██████████
  GloVe + SVM            0.4618  ████████████████
  Word2Vec + SVM         0.4629  ████████████████
  FastText + SVM         0.4716  ████████████████
  Random Forest          0.4775  ████████████████
  Logistic Regression    0.6523  ██████████████████████
  BM25 + SVM             0.7194  █████████████████████████
  TF-IDF + SVM           0.7436  ██████████████████████████

  --- Transformers ---
  DistilBERT             0.7606
  BERT-base Focal        0.8040
  Best Ensemble          0.8188


Το 0.7194 είναι λίγο χαμηλότερο από το TF-IDF+SVM (0.7419) — αναμενόμενο γιατί το BM25 είναι σχεδιασμένο για retrieval (ranking documents) και όχι για classification.
Σημαίνει ότι η length normalization του BM25 βλάπτει σε αυτό το task — τα food hazard reports έχουν παρόμοιο μήκος οπότε η normalization δεν προσφέρει κάτι, και η saturation του TF χάνει χρήσιμη πληροφορία συχνότητας.